In [19]:
import os
import pandas as pd
from pathlib import Path


In [23]:
# Base CI (dans le repo GitHub)
BASE_PATH = Path("data/data_CI/GTFS_CI")

GTFS_CLEAN_PATH = BASE_PATH
STOP_TIMES_PATH = BASE_PATH / "stop_times_small_cities.csv"

OUTPUT_BASE = BASE_PATH / "NETWORK_BASE"
OUTPUT_EDGES = BASE_PATH / "NETWORK_EDGES"

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
OUTPUT_EDGES.mkdir(parents=True, exist_ok=True)

print("BASE_PATH :", BASE_PATH.resolve())
print("STOP_TIMES_PATH existe :", STOP_TIMES_PATH.exists())


BASE_PATH : C:\Users\HP\Desktop\Transport_Recommander\data\data_CI\GTFS_CI
STOP_TIMES_PATH existe : True


In [24]:
cities = [
    d.name for d in GTFS_CLEAN_PATH.iterdir()
    if d.is_dir() and not d.name.startswith("NETWORK")
]

print("Villes détectées :", cities)
assert len(cities) > 0, "❌ Aucune ville détectée"


Villes détectées : ['Ancebus', 'Collegamenti marittimi Grimaldi', 'El Transbordador de Vizcaya', 'El Transbordador de Vizcaya (Bizkaia Bridge Ferry)', 'Rafael Nadal Coaches']


In [25]:
stop_times = pd.read_csv(
    STOP_TIMES_PATH,
    parse_dates=["arrival_time", "departure_time"],
    date_format="%Y-%m-%d %H:%M:%S"
)
print("stop_times chargé")
print("Colonnes :", stop_times.columns.tolist())
print("Nombre de lignes :", len(stop_times))


stop_times chargé
Colonnes : ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'source_folder']
Nombre de lignes : 150


In [26]:
def load_gtfs_city(city_name):
    city_path = GTFS_CLEAN_PATH / city_name

    stops = pd.read_csv(city_path / "stop_clean.txt")
    trips = pd.read_csv(city_path / "trips_clean.txt")

    return stops, trips


In [27]:
import shutil
if OUTPUT_BASE.exists():
    shutil.rmtree(OUTPUT_BASE)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
dfs = [] 
total_lignes = 0
for city in cities:
    print(f"🚀 Traitement ville : {city}")


    stops_df, trips_df = load_gtfs_city(city)

    # Filtrer stop_times pour la ville (copie explicite pour éviter SettingWithCopyWarning)
    st_city = stop_times.loc[stop_times["source_folder"] == city].copy()

    if st_city.empty:
        print(f"⚠️ Aucun stop_times pour {city}")
        continue

    if trips_df.empty:
        print(f"⚠️ Aucun trips pour {city}")
        continue

    # Harmoniser les types
    st_city["trip_id"] = st_city["trip_id"].astype(str)
    trips_df["trip_id"] = trips_df["trip_id"].astype(str)

    # Vérifier qu'il y a des trip_id communs
    common_ids = set(st_city["trip_id"]).intersection(set(trips_df["trip_id"]))
    if not common_ids:
        print(f"⚠️ Aucun trip_id commun pour {city}")
        continue

    # Jointure stop_times ↔ trips
    df = st_city.merge(
        trips_df[["trip_id", "route_id"]],
        on="trip_id",
        how="inner"
    )

    # Harmoniser stop_id
    df["stop_id"] = df["stop_id"].astype(str)
    stops_df["stop_id"] = stops_df["stop_id"].astype(str)

    # Jointure avec stops
    df = df.merge(
        stops_df[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
        on="stop_id",
        how="inner"
    )

    # Conversion des colonnes horaires en Timedelta puis format HH:MM:SS
    df["arrival_time"] = pd.to_timedelta(df["arrival_time"], errors="coerce")
    df["departure_time"] = pd.to_timedelta(df["departure_time"], errors="coerce")

    df["arrival_time"] = df["arrival_time"].dt.components.apply(
        lambda row: f"{int(row.hours):02d}:{int(row.minutes):02d}:{int(row.seconds):02d}", axis=1
    )
    df["departure_time"] = df["departure_time"].dt.components.apply(
        lambda row: f"{int(row.hours):02d}:{int(row.minutes):02d}:{int(row.seconds):02d}", axis=1
    )

    # Sauvegarde
    output_city_path = OUTPUT_BASE / city
    output_city_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_city_path / "network_base.csv", index=False)

    # Ajouter la ville et stocker dans dfs
    df["city"] = city
    dfs.append(df)

    print(f"✅ NETWORK_BASE sauvegardé pour {city} ({len(df)} lignes)")
    total_lignes += len(df)
df_all = pd.concat(dfs, ignore_index=True)

# Supprimer les doublons
df_all = df_all.drop_duplicates()

print("Total lignes NETWORK_BASE (sans doublons) :", len(df_all))




print("Total lignes NETWORK_BASE (df_all) :", len(df_all))


🚀 Traitement ville : Ancebus
✅ NETWORK_BASE sauvegardé pour Ancebus (52 lignes)
🚀 Traitement ville : Collegamenti marittimi Grimaldi
⚠️ Aucun trip_id commun pour Collegamenti marittimi Grimaldi
🚀 Traitement ville : El Transbordador de Vizcaya
✅ NETWORK_BASE sauvegardé pour El Transbordador de Vizcaya (16 lignes)
🚀 Traitement ville : El Transbordador de Vizcaya (Bizkaia Bridge Ferry)
✅ NETWORK_BASE sauvegardé pour El Transbordador de Vizcaya (Bizkaia Bridge Ferry) (16 lignes)
🚀 Traitement ville : Rafael Nadal Coaches
✅ NETWORK_BASE sauvegardé pour Rafael Nadal Coaches (6 lignes)
Total lignes NETWORK_BASE (sans doublons) : 90
Total lignes NETWORK_BASE (df_all) : 90


In [28]:
dfs = []

for city in cities:
    city_file = OUTPUT_BASE / city / "network_base.csv"
    if city_file.exists():
        df = pd.read_csv(city_file, dtype={"arrival_time": str, "departure_time": str})
        df["city"] = city

        # Conversion explicite en Timedelta
        df["arrival_time"] = pd.to_timedelta(df["arrival_time"])
        df["departure_time"] = pd.to_timedelta(df["departure_time"])

        dfs.append(df)

assert len(dfs) > 0, "❌ Aucun NETWORK_BASE chargé"

df_all = pd.concat(dfs, ignore_index=True).drop_duplicates()

print("Total lignes NETWORK_BASE (sans doublons) :", len(df_all))


Total lignes NETWORK_BASE (sans doublons) : 90


In [29]:
# 1️⃣ Ordonner correctement
df_all = df_all.sort_values(["trip_id", "stop_sequence"])

# 2️⃣ Calcul du stop suivant (edge direction)
df_all["to_stop_id"] = df_all.groupby("trip_id")["stop_id"].shift(-1)
df_all["to_lat"] = df_all.groupby("trip_id")["stop_lat"].shift(-1)
df_all["to_lon"] = df_all.groupby("trip_id")["stop_lon"].shift(-1)
df_all["to_arrival_time"] = df_all.groupby("trip_id")["arrival_time"].shift(-1)

# 3️⃣ Supprimer la dernière ligne de chaque trip
edges = df_all.dropna(subset=["to_stop_id"]).copy()

# 4️⃣ Temps de parcours
edges["travel_time_sec"] = (
    edges["to_arrival_time"] - edges["arrival_time"]
).dt.total_seconds()

# 5️⃣ Deltas géographiques
edges["delta_lat"] = edges["to_lat"] - edges["stop_lat"]
edges["delta_lon"] = edges["to_lon"] - edges["stop_lon"]

# 6️⃣ NORMALISATION (clé pour le CI)
edges = edges.rename(columns={
    "stop_id": "from_stop_id",
    "stop_lat": "from_lat",
    "stop_lon": "from_lon"
})

# 7️⃣ Dataset final CI aligné FULL DATA
edges_final = edges[[
    "city",
    "trip_id",
    "route_id",
    "from_stop_id",
    "to_stop_id",
    "from_lat",
    "from_lon",
    "to_lat",
    "to_lon",
    "travel_time_sec",
    "delta_lat",
    "delta_lon"
]]

print("✅ Edges CI créés :", len(edges_final))
print("Colonnes :", list(edges_final.columns))


✅ Edges CI créés : 74
Colonnes : ['city', 'trip_id', 'route_id', 'from_stop_id', 'to_stop_id', 'from_lat', 'from_lon', 'to_lat', 'to_lon', 'travel_time_sec', 'delta_lat', 'delta_lon']


In [30]:
for city in edges_final["city"].unique():
    city_edges = edges_final[edges_final["city"] == city]

    safe_city = city.replace("/", "_").replace(" ", "_")
    out_path = OUTPUT_EDGES / safe_city
    out_path.mkdir(parents=True, exist_ok=True)

    city_edges.to_csv(out_path / "edges.csv", index=False)

    print(f"✅ Edges sauvegardés pour {city} ({len(city_edges)} lignes)")


✅ Edges sauvegardés pour El Transbordador de Vizcaya (16 lignes)
✅ Edges sauvegardés pour El Transbordador de Vizcaya (Bizkaia Bridge Ferry) (8 lignes)
✅ Edges sauvegardés pour Rafael Nadal Coaches (4 lignes)
✅ Edges sauvegardés pour Ancebus (46 lignes)


In [31]:
# Vérifications minimales CI
assert edges_final["travel_time_sec"].isna().sum() == 0
assert (edges_final["travel_time_sec"] >= 0).all()
assert edges_final["city"].nunique() >= 3

print("✅ Vérifications CI OK – logique métier validée")


✅ Vérifications CI OK – logique métier validée


In [32]:
OUTPUT_DIR = "ci_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Garder seulement 50 000 lignes max
edges_ci_pd = edges_final.head(50000)

edges_ci_pd.to_csv(f"{OUTPUT_DIR}/edges_ci_sample.csv", index=False)

print("✅ Échantillon CI exporté en CSV")


✅ Échantillon CI exporté en CSV
